<a href="https://colab.research.google.com/github/zpsheldon/meg-neural-decoding/blob/main/calc_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Imports

In [1]:
# Install additional depdendencies
%pip install -q lightning torchmetrics scikit-learn plotly ipywidgets pnpl

# Set up base path for dataset and related files (base_path is assumed to be set in the cells below!)
base_path = "./libribrain"
try:
    import google.colab  # This module is only available in Colab.
    in_colab = True
    base_path = "/content"  # This is the folder displayed in the Colab sidebar
except ImportError:
    in_colab = False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.7 MB/s eta 0:00:00


In [2]:
from pnpl.datasets import LibriBrainSpeech
from torch.utils.data import DataLoader
import pandas as pd
import random
import torch
import platform

## Load data

In [3]:
num_books = 7
num_chapters = [9, 12, 12, 12, 15, 14, 14]

In [ ]:
# Conditionally set num_workers to avoid multiprocessing issues (try increasing if performance is problematic)
num_workers = 2 if in_colab else 0

run_keys = [("0",str(i),f"Sherlock{j}","1") for j in range(1,7) for i in range(1, num_chapters[j-1])]
all_data = LibriBrainSpeech(
  data_path=f"{base_path}/data/",
  include_run_keys = run_keys,
  tmin=0.0,
  tmax=0.8,
  preload_files = True
)

(…)-0_ses-3_task-Sherlock4_run-1_events.tsv: 0.00B [00:00, ?B/s]

(…)-0_ses-1_task-Sherlock5_run-1_events.tsv: 0.00B [00:00, ?B/s]

(…)-0_ses-5_task-Sherlock5_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock5/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/426M [00:00<?, ?B/s]

Sherlock2/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/391M [00:00<?, ?B/s]

...

(…)-0_ses-2_task-Sherlock6_run-1_events.tsv: 0.00B [00:00, ?B/s]

(…)0_ses-11_task-Sherlock2_run-1_events.tsv: 0.00B [00:00, ?B/s]

(…)-0_ses-2_task-Sherlock2_run-1_events.tsv: 0.00B [00:00, ?B/s]

(…)-0_ses-5_task-Sherlock6_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock4/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/818M [00:00<?, ?B/s]

## Load tsv event files and calculate class balances

In [4]:
run_keys = [("0",1,"Sherlock1","1")]
all_data = LibriBrainSpeech(
  data_path=f"{base_path}/data/",
  include_run_keys = run_keys,
  tmin=0.0,
  tmax=0.8,
  preload_files = True
)
tsv_file_path = f"{base_path}/data/Sherlock1/derivatives/events/sub-0_ses-1_task-Sherlock1_run-1_events.tsv"
data = pd.read_csv(tsv_file_path, sep='\t')
data

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


(…)-0_ses-1_task-Sherlock1_run-1_events.tsv: 0.00B [00:00, ?B/s]

Sherlock1/derivatives/serialised/sub-0_s(…):   0%|          | 0.00/342M [00:00<?, ?B/s]

Done!
calculated stats for:  ('0', 1, 'Sherlock1', '1')


,idx,wavile,kind,segment,sentenceidx,wordidx,phonemeidx,timemeg,timeds,timechapter,timesentence,duration
0,1,segments/,silence,NaN,NaN,NaN,NaN,28.772,28.772,0.000,NaN,1.188
1,2,segments/,word,A,0.0,0.0,NaN,30.086,30.084,1.298,0.11,0.100
2,3,segments/,phoneme,ah_S,0.0,0.0,0.0,30.086,30.084,1.298,0.11,0.100
3,4,segments/,word,Study,0.0,1.0,NaN,30.186,30.184,1.398,0.21,0.370
4,5,segments/,phoneme,s_B,0.0,1.0,0.0,30.186,30.184,1.398,0.21,0.080
...,...,...,...,...,...,...,...,...,...,...,...,...
12801,12802,segments/,phoneme,er_E,187.0,2.0,4.0,1098.278,1098.276,1064.404,0.79,0.060
12802,12803,segments/,word,one,187.0,3.0,NaN,1098.338,1098.336,1064.464,0.85,0.340
12803,12804,segments/,phoneme,w_B,187.0,3.0,0.0,1098.338,1098.336,1064.464,0.85,0.120
12804,12805,segments/,phoneme,ah_I,187.0,3.0,1.0,1098.458,1098.460,1064.584,0.97,0.100


In [ ]:
tsv_file_paths = [f"{base_path}/data/Sherlock1/derivatives/events/sub-0_ses-{i}_task-Sherlock{j}_run-1_events.tsv" for j in range(1,7) for i in range(1, num_chapters[j-1])]

phoneme_n = {}
silence_n = {}
speech_n = {}
for f in tsv_file_paths:
  data = pd.read_csv(tsv_file_path, sep='\t')
  group_df = data.groupby(by="kind",as_index=False).count()
  curr_phoneme_n = group_df.iloc[0]["idx"]
  curr_silence_n = group_df.iloc[1]["idx"]
  curr_speech_n = group_df.iloc[2]["idx"]
  phoneme_n[f] = curr_phoneme_n
  silence_n[f] = curr_silence_n
  speech_n[f] = curr_speech_n